# Домашнє завдання 4. DML та DDL команди. Складні SQL-вирази

**Датасет:** [Olist Brazilian E-commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) —
~100 тис. замовлень бразильського маркетплейсу Olist за 2016–2018 роки.

Використано 6 таблиць: `olist_customers`, `olist_orders`, `olist_order_items`, `olist_products`, `olist_sellers`, `olist_order_reviews`.
CSV спочатку завантажуються в raw-таблиці (`*_raw`), а потім переносяться в typed-таблиці з ключами та constraints.

## Завдання 1. Setup і typed-схема Olist

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub tzdata

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import tempfile
from pathlib import Path

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text


def ensure_pg_timezone_db():
    """У збірці pgserver для Windows немає tz-бази, тому копіюємо її з пакета tzdata (у Colab нічого не робить)."""
    import shutil
    import tzdata

    tz_dir = Path(pgserver.__file__).parent / "pginstall" / "share" / "postgresql" / "timezone"
    if (tz_dir / "Europe" / "Kyiv").is_file():
        return
    try:
        shutil.copytree(Path(tzdata.__file__).parent / "zoneinfo", tz_dir, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("__init__.py", "__pycache__", "*.pyc"))
    except OSError as e:
        raise RuntimeError(
            f"Не вдалося записати tz-базу в {tz_dir} ({e}). "
            "Встановіть залежності у віртуальне середовище (python -m venv .venv), а не в системний Python."
        ) from e


ensure_pg_timezone_db()

# у Colab tempfile.gettempdir() = /tmp, тобто шлях /tmp/hw4_pg як у завданні
pg = pgserver.get_server(os.path.join(tempfile.gettempdir(), 'hw4_pg'), cleanup_mode='stop')
# явно вказуємо драйвер psycopg2 (новіша SQLAlchemy за замовчуванням шукає psycopg 3)
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://'), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit


In [3]:
pd.set_option('display.max_columns', None)


def run(sql):
    """DDL / DML без результату."""
    with engine.begin() as conn:
        conn.execute(text(sql))


def show(sql):
    """SELECT або DML з RETURNING -> DataFrame (зміни комітяться)."""
    with engine.begin() as conn:
        res = conn.execute(text(sql))
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

### Отримання файлів Olist

Якщо CSV лежать у репозиторії в папці `data/olist/` — беремо їх звідти (варіант B).
Інакше файли завантажуються з Kaggle через `kagglehub` (варіант A, для публічного датасету в Colab працює без логіну).

In [4]:
if os.path.exists('data/olist/olist_orders_dataset.csv'):
    # Варіант B: CSV вже лежать у репозиторії
    dataset_dir = 'data/olist'
else:
    # Варіант A: завантаження з Kaggle
    import kagglehub
    dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')

print(dataset_dir)

data/olist


### Raw-таблиці

In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### Typed-таблиці

Спочатку видаляємо всі таблиці (щоб notebook можна було запускати повторно), потім створюємо typed-схему з PK, FK, CHECK і NOT NULL.
Для відгуків первинним ключем є технічний `review_row_id`, а `review_id` зберігається як звичайна колонка, бо в Olist він може повторюватися.

In [6]:
run("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")

Переносимо дані з raw у typed через `INSERT ... SELECT`. У CSV колонки товарів називаються `product_name_lenght` / `product_description_lenght` (з помилкою), тому в SELECT звертаємось саме до них.

In [7]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

### Smoke-test кількості рядків

In [8]:
show("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


**Висновок.** Кількість рядків у typed-таблицях така сама, як у raw (див. вивід завантаження CSV):
99 441 клієнтів, 99 441 замовлень, 112 650 позицій, 32 951 товар, 3 095 продавців і 99 224 відгуки.
Тобто жоден рядок не загубився і не був відкинутий через PK / FK / CHECK.

Перевірка, чому `review_id` не можна робити primary key:

In [9]:
show("""
SELECT
    COUNT(*)                  AS n_reviews,
    COUNT(DISTINCT review_id) AS n_unique_review_id,
    COUNT(*) FILTER (WHERE review_comment_message IS NULL) AS n_empty_message
FROM olist_order_reviews;
""")

,n_reviews,n_unique_review_id,n_empty_message
0,99224,98410,58247


**Висновок.** На 99 224 відгуки є лише 98 410 унікальних `review_id`, тобто частина `review_id` повторюється.
Якби `review_id` був PRIMARY KEY, перенесення з raw впало б з помилкою, тому використано технічний `review_row_id`.
Також видно, що більше половини відгуків (58 247) не мають тексту коментаря — є тільки оцінка.

## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. INSERT ... RETURNING: alert для продавця з найбільшою кількістю запізнень

In [10]:
run("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [11]:
show("""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
""")

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-10-11 20:57:30.211850+03:00


**Пояснення.** Запит знаходить продавця з найбільшою кількістю запізнених замовлень (172 замовлення) і вставляє для нього alert.
Завдяки `RETURNING` ми одразу бачимо, що саме вставилось: згенерований `alert_id`, продавця, тип і `created_at` —
не треба робити окремий SELECT. Такий вивід можна зберігати як audit-log: що і коли було створено.

Цей INSERT сам по собі не ідемпотентний — повторний запуск тільки цієї клітинки додасть ще один такий самий alert.
Тому на початку блоку таблиця `seller_alerts` перестворюється (`DROP TABLE IF EXISTS`), і після Restart & Run All alert буде один.

### 2.2. INSERT ... ON CONFLICT DO UPDATE: upsert seller_score

In [12]:
run("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [13]:
seller_score_sql = """
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
"""

df = show(seller_score_sql)
print('повернуто рядків:', len(df))
df.head(10)

повернуто рядків: 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-11 20:57:30.741430+03:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-11 20:57:30.741430+03:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-11 20:57:30.741430+03:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-11 20:57:30.741430+03:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-11 20:57:30.741430+03:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-11 20:57:30.741430+03:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-11 20:57:30.741430+03:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-11 20:57:30.741430+03:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-11 20:57:30.741430+03:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-11 20:57:30.741430+03:00


Запускаємо той самий upsert ще раз і перевіряємо, що кількість рядків у `seller_score` не змінилась:

In [14]:
df = show(seller_score_sql)
print('повернуто рядків після повторного запуску:', len(df))

show("""
SELECT
    COUNT(*)                   AS n_rows,
    COUNT(DISTINCT seller_id)  AS n_sellers,
    COUNT(*) FILTER (WHERE avg_rating IS NULL) AS n_without_reviews
FROM seller_score;
""")

повернуто рядків після повторного запуску: 3095


,n_rows,n_sellers,n_without_reviews
0,3095,3095,5


In [15]:
show("""
SELECT tier, COUNT(*) AS n_sellers
FROM seller_score
GROUP BY tier
ORDER BY n_sellers DESC;
""")

,tier,n_sellers
0,silver,1568
1,gold,983
2,bronze,539
3,NaN,5


**Пояснення.**

- **Чому `SELECT DISTINCT` у `seller_orders`.** В одному замовленні у продавця може бути кілька позицій. `DISTINCT` залишає одну пару
  `seller_id × order_id`, щоб одне замовлення рахувалось для продавця один раз.
- **Чому review спочатку агрегуємо до `order_id`.** До одного замовлення може бути кілька відгуків. `review_per_order` дає рівно одну
  оцінку на замовлення, і після join з `seller_orders` нічого не множиться. Якби join-ити відгуки напряму з `olist_order_items`,
  відгук до замовлення з 3 позиціями порахувався б 3 рази.
- **Чому `ON CONFLICT (seller_id) DO UPDATE` робить запуск повторюваним.** `seller_id` — primary key. При повторному запуску
  рядок вже існує, тому замість помилки або дубліката він просто оновлюється. Це видно вище: після другого запуску в таблиці
  так само 3 095 рядків і 3 095 унікальних продавців.
- **Чому є `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`.** Через `LEFT JOIN` продавець залишається, навіть якщо до його замовлень
  немає жодного відгуку. Тоді `AVG` повертає NULL, `COUNT(rpo.order_id)` — 0, а CASE дає NULL. Таких продавців 5.

**Інтерпретація.** Більшість продавців мають рівень silver (1 568), gold — 983, bronze — 539.
Обмеження: відгук в Olist прив'язаний до замовлення, а не до продавця. Якщо в замовленні товари кількох продавців,
одна й та сама оцінка потрапляє кожному з них — це навчальне спрощення.

### 2.3. UPDATE ... FROM: прапорець запізнення в olist_orders

In [16]:
run("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")

In [17]:
df = show("""
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
""")
print('оновлено рядків:', len(df))
print(df['is_late'].value_counts())
df.head()

оновлено рядків: 99441
is_late
False    92906
True      6535
Name: count, dtype: int64


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False


**Пояснення.** `UPDATE ... FROM` бере значення з CTE `delivery_flags` і з'єднує його з `olist_orders` за `order_id` (один до одного,
бо `order_id` — PK). Оновлено всі 99 441 замовлення: 6 535 з них доставлені пізніше очікуваної дати (`is_late = TRUE`).

Запит ідемпотентний: `is_late` щоразу перераховується з дат, а не просто ставиться в TRUE, тому повторний запуск дасть той самий результат.
`ADD COLUMN IF NOT EXISTS` теж не впаде при повторному запуску.

### 2.4. DELETE ... RETURNING: видалення застарілих alert-ів

In [18]:
show("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")

,alert_id,seller_id,alert_type,created_at


**Пояснення.** Результат порожній, і це нормально: єдиний alert створено кілька секунд тому в п. 2.1, тому alert-ів старших за 30 днів немає.
Якби вони були, `RETURNING` показав би, які саме рядки видалено, — це зручно для перевірки і для audit-логу.
Запит безпечний для повторного запуску: видаляються тільки рядки за умовою `WHERE`, а повторний запуск нічого зайвого не видалить.

## Завдання 3. CREATE TABLE customer_segments

Спочатку створюємо дедуплікований customer dimension: один рядок на `customer_unique_id`, з `customer_id` останнього замовлення.

In [19]:
run("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")

show("""
SELECT
    (SELECT COUNT(*) FROM olist_customers)    AS n_customer_id,
    (SELECT COUNT(*) FROM olist_customer_dim) AS n_customer_unique_id;
""")

,n_customer_id,n_customer_unique_id
0,99441,96096


Таблиця `customer_segments` з identity PK, FK, UNIQUE, CHECK, NOT NULL і DEFAULT:

In [20]:
run("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

Наповнення через `INSERT ... SELECT ... ON CONFLICT`. `reference_date` = день після останньої покупки в датасеті.

In [21]:
df = show("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
""")
print('повернуто рядків:', len(df))
print(df['segment_name'].value_counts())
df.head()

повернуто рядків: 96096
segment_name
regular     45982
inactive    40032
new          8891
VIP          1191
Name: count, dtype: int64


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1


Перевірка constraints:

In [22]:
show("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


**Результат.** У `customer_segments` 96 096 рядків — рівно стільки, скільки унікальних покупців у `olist_customer_dim`
(при цьому `customer_id` — 99 441). Розподіл сегментів: regular — 45 982, inactive — 40 032, new — 8 891, VIP — 1 191.
Перевірка `pg_constraint` показує всі обмеження: PK (`p`), UNIQUE (`u`), FK (`f`) і CHECK (`c`).

**Пояснення.**

- **Чому `customer_unique_id`.** В Olist `customer_id` створюється під кожне замовлення, тому один реальний покупець може мати кілька
  `customer_id`. Якщо рахувати сегменти по `customer_id`, повторні покупки розбились би на різні рядки і `n_orders` майже завжди був би 1.
- **Навіщо `olist_customer_dim`.** В `olist_customers` значення `customer_unique_id` не унікальне, тому на нього не можна зробити FK.
  У `olist_customer_dim` воно є PRIMARY KEY, і `customer_segments` посилається саме на неї.
- **Чому `reference_date` з датасету.** Дані закінчуються у 2018 році. З `CURRENT_DATE` усі покупці мали б recency у тисячі днів, сегмент
  `new` був би порожній, а результат змінювався б залежно від дня запуску. Дата `MAX(order_purchase_timestamp) + 1` завжди однакова.
- **Як працює `ON CONFLICT`.** На `customer_unique_id` є UNIQUE, тому при повторному запуску існуючі рядки оновлюються
  (`DO UPDATE`), а не дублюються.

## Завдання 4. JOIN-запити

### 4.1. INNER JOIN: топ категорій за revenue і штатом продавця

**Business-question:** які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [23]:
show("""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Інтерпретація.** Увесь топ-10 — це продавці зі штату SP (Сан-Паулу), тобто основний revenue маркетплейсу генерують продавці з SP.
Найбільше заробляє категорія `cama_mesa_banho` (постільна білизна / ванна) — 1,07 млн BRL з 8 214 замовлень.
`relogios_presentes` (годинники / подарунки) має майже такий самий revenue (1,02 млн), але майже вдвічі менше замовлень, тобто
там дорожчі товари. Review тут не використовується, тому join з `olist_order_items` нічого не множить.

### 4.2. LEFT JOIN + COALESCE: customer-level features

**Business-question:** які customer-level features можна побудувати для реального покупця на рівні `customer_unique_id`?

In [24]:
show("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


**Інтерпретація.** Для кожного покупця отримали ознаки `n_orders`, `total_spend` і `avg_review_score`. Найбільше витратив покупець з RJ —
13 664 BRL за одне замовлення. Більшість топ-покупців мають лише 1 замовлення, повторних мало (наприклад, покупець з RS має 4 замовлення).

CTE `review_per_order` спочатку зводить відгуки до одного значення на `order_id`, тому оцінка не множиться на кількість позицій у замовленні.
Мінус `COALESCE(..., 0)`: у покупця без відгуку `avg_review_score = 0.00` (наприклад, 5-й рядок), хоча оцінки 0 не існує —
для ML краще додати окрему ознаку «є відгук / немає відгуку».

### 4.3. FULL OUTER JOIN: штати клієнтів і продавців

**Business-question:** які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [25]:
show("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Інтерпретація.** Всього 27 штатів. У 23 штатах є і клієнти, і продавці (`both`). 4 штати (AL, AP, RR, TO) мають тільки клієнтів —
продавців там немає, тому товари туди завжди везуть з інших штатів. Штатів тільки з продавцями немає (`sellers_only` відсутній).
Продавці дуже сконцентровані в SP: 1 849 з 3 095.

### 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст

**Business-question:** які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [26]:
show("""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Інтерпретація.** Запит знаходить пари продавців з одного штату, але з різних міст (перші 20 — штат BA). Умова `s1.seller_id < s2.seller_id`
прибирає пару продавця з самим собою і дзеркальні дублікати (A, B) / (B, A). Такі пари можна використати, щоб порівнювати продавців
всередині регіону (наприклад, столицю `salvador` з меншими містами). Також видно проблему якості даних: у одного продавця місто записано
як `bahia`, хоча це назва штату.

### 4.5. Anti-join: замовлення без відгуку

**Business-question:** які замовлення не мають жодного review-запису?

In [27]:
show("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Інтерпретація.** У датасеті є замовлення без жодного відгуку. Серед найстаріших такі замовлення мають статуси `shipped` та `unavailable`
(покупець не отримав товар, тому й не залишив відгук), але більшість — `delivered`, тобто частина покупців просто не ставить оцінку.
Для аналізу рейтингу це означає, що відсутність відгуку не можна прирівнювати до поганої оцінки.
Перевіряється `rv.review_row_id IS NULL`, бо це PK правої таблиці і в реальному рядку він ніколи не буває NULL.

## Завдання 5. GROUP BY, HAVING і агрегати

### 5.1. Revenue по категоріях товарів (HAVING по revenue)

**Business-question:** які категорії приносять більше 10 000 BRL виручки за ціною товарів?

In [28]:
show("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Інтерпретація.** Умову `SUM(price) > 10 000` пройшли 53 категорії. Найбільша виручка — `beleza_saude` (1,26 млн BRL) і
`relogios_presentes` (1,21 млн), а найбільше замовлень — у `cama_mesa_banho` (9 417). У `pcs` (комп'ютери) мало замовлень (181),
але найвища середня ціна товару — 1 098 BRL. Окремо видно групу `None` — товари без категорії (1 451 замовлення), це пропуски в даних.

### 5.2. AOV по штатах покупців (тільки штати з ≥ 100 замовлень)

**Business-question:** у яких штатах середній чек замовлення (AOV) найвищий?

In [29]:
show("""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Інтерпретація.** HAVING `COUNT(*) >= 100` залишив 24 штати з 27 (відпали AC, AP, RR — там замало замовлень, щоб AOV був надійним).
Найвищий середній чек у PB (265 BRL) та інших північних і північно-східних штатах, найнижчий — у SP (143 BRL), хоча SP дає найбільший GMV (5,9 млн).
Ймовірно, в далеких штатах дорожча доставка (`freight_value` входить у `order_total`) і купують рідше, але дорожчі товари.
CTE `order_totals` спочатку рахує суму на рівні замовлення, тому `COUNT(*)` рахує саме замовлення, а не позиції.

### 5.3. Категорії, які купував покупець (STRING_AGG)

**Business-question:** які категорії купують покупці, що зробили 2 і більше замовлень, і скільки вони витратили?

In [30]:
show("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


**Інтерпретація.** Тут HAVING `COUNT(DISTINCT o.order_id) >= 2` залишає тільки покупців з повторними замовленнями.
`STRING_AGG(DISTINCT ...)` збирає їхні категорії в один рядок. Видно, що більшість повторних покупців купують в одній і тій самій категорії
(`pcs`, `telefonia`, `relogios_presentes`), і лише деякі — у двох різних. `total_spend` тут рахується тільки по `price` без доставки,
тому для покупця `da122...` він менший, ніж у запиті 4.2 (7 388 проти 7 571.63).

### Різниця між WHERE і HAVING

На прикладі запиту 5.1: умову `HAVING SUM(oi.price) > 10000` не можна написати у `WHERE`, бо `WHERE` виконується **до** групування —
він фільтрує окремі рядки `olist_order_items`, і суми по категорії на цьому етапі ще не існує.
`HAVING` виконується **після** `GROUP BY` і відкидає вже цілі групи (категорії). Якби треба було, наприклад, врахувати тільки позиції
з ціною більше 0, це була б умова `WHERE oi.price > 0` — вона стосується одного рядка, а не групи.
Так само в 4.1 `WHERE o.order_status = 'delivered'` відкидає рядки до агрегації.

## Завдання 6. Set operations, window function і Reflection

### 6.1. UNION: усі штати, що є у клієнтів або продавців

In [31]:
show("""
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
""")

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


**Інтерпретація.** Обрано `UNION`, бо потрібен список унікальних пар «штат — роль»: клієнтів у кожному штаті тисячі,
і `UNION ALL` повернув би 99 441 + 3 095 рядків. `UNION` прибрав дублікати і залишив 50 рядків.
Колонка `role` потрібна, щоб розрізнити, звідки рядок; штати без продавців (AL, AP, RR, TO) мають тільки рядок `customer`.

### 6.2. INTERSECT: штати, у яких є і клієнти, і продавці

In [32]:
show("""
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


**Інтерпретація.** `INTERSECT` повертає тільки штати, які є в обох наборах, — їх 23. Це той самий результат, що й `both` у FULL OUTER JOIN (4.3),
але записаний коротше. INTERSECT обрано, бо питання саме про перетин двох множин.

### 6.3. Window function: найновіше замовлення кожного покупця через ROW_NUMBER()

In [33]:
show("""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


**Інтерпретація.** `ROW_NUMBER()` нумерує замовлення кожного покупця (`PARTITION BY customer_unique_id`) від найновішого до найстарішого,
а `WHERE rn = 1` залишає одне останнє замовлення на покупця. На відміну від `GROUP BY`, так можна отримати не тільки дату,
а й весь рядок — `order_id` і `customer_id` цього замовлення. Таке ж правило «останній customer_id» використано в `olist_customer_dim`.
Для ML це зручно, щоб брати дату останньої покупки (recency) або останню адресу покупця.

## Reflection

**Найскладніший JOIN.** Для мене найскладнішим був FULL OUTER JOIN. Спочатку треба окремо агрегувати штати клієнтів і продавців,
а потім через `COALESCE` зібрати одну колонку `state`, інакше рядки «тільки справа» мали б порожній штат.

**Business-question.** Найважче було сформулювати питання для SELF JOIN: пари продавців самі по собі мало що кажуть бізнесу,
тому я пояснив їх як кандидатів для порівняння всередині регіону.

**Корисна PostgreSQL-фіча.** Для production-ETL найкорисніший `ON CONFLICT`: upsert `seller_score` і `customer_segments`
можна запускати багато разів без дублікатів. `RETURNING` теж зручний, бо одразу видно змінені рядки.

**Window functions у ML.** `ROW_NUMBER` — останнє замовлення покупця, `LAG` — кількість днів між покупками,
`RANK` — рейтинг продавців у штаті, ковзне середнє продажів по днях для прогнозування.

**Data quality.** Повтори `review_id` (98 410 унікальних на 99 224 записи), більше половини відгуків без тексту,
замовлення без відгуків, товари без категорії (`None` у 5.1), місто `bahia` у продавця, 5 продавців без жодного відгуку.

**customer_unique_id.** `customer_id` створюється під кожне замовлення, тому один покупець може мати кілька таких id
(99 441 `customer_id` проти 96 096 покупців). З `customer_id` frequency майже завжди дорівнювала б 1.

**Грануляність seller_id × order_id.** Відгук прив'язаний до замовлення, а в замовленні може бути кілька позицій одного продавця.
Якщо join-ити відгуки з `order_items` напряму, один відгук рахувався б кілька разів і середня оцінка продавця була б спотворена.